### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
# Habilitar si se ejecuta desde Colab

#%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [157]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [158]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [159]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [160]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [161]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [162]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [163]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [164]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [165]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [166]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [167]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [168]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [169]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [170]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [171]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [172]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  4703, 10870,  4333], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [173]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [174]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [175]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [176]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [177]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [178]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


---
### Respuestas:

1. Vectorizar documentos

In [179]:
def random_documents_idx(documents, n=5, seed=42):
    np.random.seed(seed)
    return np.random.choice(len(documents), size=n, replace=False)

def vectorize_documents(documents):
    vect = TfidfVectorizer()
    vectors = vect.fit_transform(documents)
    return vectors, vect

def most_similar_documents(document_idx, vectors, top_k=5):
    cossim = cosine_similarity(vectors[document_idx], vectors)[0]
    mostsim = np.argsort(cossim)[::-1][1:top_k+1]
    return mostsim, cossim

def document_type(document_idx, documents):
    return documents.target_names[documents.target[document_idx]]

In [180]:
documents = newsgroups_train

random_idx = random_documents_idx(documents.data)

vectors, _ = vectorize_documents(documents.data)

for idx in random_idx:
    most_similar_idxs, cossim = most_similar_documents(idx, vectors)
    print(f"DOCUMENTO #{idx} - TIPO '{document_type(idx, documents)}'")
    print("\t>>>>>>>> CONTENIDO >>>>>>>>>")
    print(f"{documents.data[idx][:300]}...")
    print("\t<<<<<<<<<<<<<<<<<<<<<<<<<<<<<")

    print("DOCUMENTOS SIMILARES:")
    for similar_idx in most_similar_idxs:
        print(f"\tDOCUMENTO #{similar_idx} - TIPO: '{document_type(similar_idx, documents)}' - SIMILITUD: {cossim[similar_idx]:.3f}")
    print("\n")

DOCUMENTO #7492 - TIPO 'comp.sys.mac.hardware'
	>>>>>>>> CONTENIDO >>>>>>>>>
Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging one's head
rrn@po.cwru.edu  |  against a wall...with less opportunity for reward" ...
	<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
DOCUMENTOS SIMILARES:
	DOCUMENTO #10935 - TIPO: 'comp.sys.mac.hardware' - SIMILITUD: 0.667
	DOCUMENTO #7258 - TIPO: 'comp.sys.ibm.pc.hardware' - SIMILITUD: 0.348
	DOCUMENTO #4971 - TIPO: 'comp.sys.mac.hardware' - SIMILITUD: 0.180
	DOCUMENTO #4303 - TIPO: 'misc.forsale' - SIMILITUD: 0.155
	DOCUMENTO #645 - TIPO: 'comp.sys.mac.hardware' - SIMILITUD: 0.141


DOCUMENTO #3546 - TIPO 'comp.os.ms-windows.misc'
	>>>>>>>> CONTENIDO >>>>>>>>>


     Don't bother if you have CPBackup or Fastback.  They all offer options 
not available in the stripped-down MS version (FROM CPS!).  Examples - no 
proprietary form

📝 Observación: Para los casos analizados se observa que la mayoria comparten las etiquetas (o estan muy relacionados) con los documentos mas similares. Aunque el mayor desvio se da para el documento #3813 del tipo 'rec.sport.hockey', en este caso los documentos relacionados tienen etiquetas como 'alt.atheism, soc.religion.christian' lo que indica que no hay documentos parecidos.

2. Construir un modelo de clasificación por prototipos (tipo zero-shot)

In [181]:
def predict_by_prototype(X_train, y_train, X_test):
    cossim = cosine_similarity(X_test, X_train)
    most_similar = np.argmax(cossim, axis=1)
    y_pred_proto = y_train[most_similar]
    return y_pred_proto

def evaluate_predictions(y_test, y_pred):
    score = f1_score(y_test, y_pred, average='macro')
    print(f"F1-Score(macro): {score:.3f}")

y_pred = predict_by_prototype(X_train, y_train, X_test)
evaluate_predictions(y_test, y_pred)

F1-Score(macro): 0.505


📝 Observación: Si bien el modelo de clasificación obtuvo un valor bajo de F1-score, para ser un modelo que no tuvo entrenamiento específico para las clases (20 categorias) su rendimiento es aceptable. Dicho modelo puede servir como base de comparación (baseline).

3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

In [182]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pandas as pd

# 1. Cargar datos
train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

X_train, y_train = train.data, train.target
X_test, y_test = test.data, test.target

# 2. Definición del Pipeline
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('model', MultinomialNB())
])

# 3. Grid de parámetros
param_grid = [
    {
        'vectorizer': [TfidfVectorizer()],
        'vectorizer__ngram_range': [(1,1), (1,2)],
        'vectorizer__max_df': [0.8, 1.0],
        'vectorizer__min_df': [1, 2],
        'vectorizer__sublinear_tf': [True, False],
        'model': [MultinomialNB()],
        'model__alpha': [0.01, 0.1, 0.5, 1.0]
    },
    {
        'vectorizer': [TfidfVectorizer()],
        'vectorizer__ngram_range': [(1,1), (1,2)],
        'vectorizer__sublinear_tf': [True, False],
        'model': [ComplementNB()],
        'model__alpha': [0.01, 0.1, 0.5, 1.0]
    },
    {
        'vectorizer': [CountVectorizer()],
        'vectorizer__ngram_range': [(1,1), (1,2)],
        'model': [MultinomialNB(), ComplementNB()],
        'model__alpha': [0.01, 0.1, 0.5]
    }
]

# 4. Entrenamiento
grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

# 5. Mejor modelo
print(f"Mejores parámetros: {grid.best_params_}")

Fitting 3 folds for each of 92 candidates, totalling 276 fits
Mejores parámetros: {'model': ComplementNB(), 'model__alpha': 0.1, 'vectorizer': TfidfVectorizer(), 'vectorizer__ngram_range': (1, 2), 'vectorizer__sublinear_tf': True}


In [183]:
# 6. Comparación de resultados
df_results = pd.DataFrame(grid.cv_results_)
params_df = df_results['params'].apply(pd.Series)
df_full = pd.concat([df_results, params_df], axis=1)
top_k = df_full.sort_values(by='mean_test_score', ascending=False).head(15)

top_k[[
    'mean_test_score',
    'model',
    'model__alpha',
    'vectorizer',
    'vectorizer__ngram_range',
    'vectorizer__sublinear_tf'
]]

,mean_test_score,model,model__alpha,vectorizer,vectorizer__ngram_range,vectorizer__sublinear_tf
70,0.764411,ComplementNB(),0.10,TfidfVectorizer(),"(1, 2)",True
71,0.763753,ComplementNB(),0.10,TfidfVectorizer(),"(1, 2)",False
68,0.754561,ComplementNB(),0.10,TfidfVectorizer(),"(1, 1)",True
72,0.754048,ComplementNB(),0.50,TfidfVectorizer(),"(1, 1)",True
69,0.753392,ComplementNB(),0.10,TfidfVectorizer(),"(1, 1)",False
67,0.751825,ComplementNB(),0.01,TfidfVectorizer(),"(1, 2)",False
73,0.751081,ComplementNB(),0.50,TfidfVectorizer(),"(1, 1)",False
75,0.748693,ComplementNB(),0.50,TfidfVectorizer(),"(1, 2)",False
66,0.747874,ComplementNB(),0.01,TfidfVectorizer(),"(1, 2)",True
74,0.747693,ComplementNB(),0.50,TfidfVectorizer(),"(1, 2)",True


In [184]:
# 7. Evaluación final
y_pred = grid.predict(X_test)
f1 = f1_score(y_test, y_pred, average='macro')

print(f"F1-score final: {f1:.3f}")

F1-score final: 0.706


📝 Observación: Se utilizó GridSearchCV para la búsqueda de hiperparámetros ya que facilita la comparación de modelos con distintas configuraciones. Los resultados obtenidos muestran que el modelo ComplementNB presenta un mejor desempeño en comparación con MultinomialNB. Otra cuestion a destacar es que el vectorizar CountVectorizer no se encuentra entre los mejores modelos, lo que sugiere que la vectorización TF-IDF es más efectiva para este conjunto de datos específico.
Por último, el valor de F1-score (0.706) obtenido en este experimento fue mejor que el obtenido con el modelo de clasificación por prototipos (0.505), lo cual era de esperarse.

4. Transponer la matriz documento-término.

In [185]:
def most_similar_words(word_idx, vectors, top_k=5):
    cossim = cosine_similarity(vectors[word_idx], vectors)[0]
    mostsim = np.argsort(cossim)[::-1][1:top_k+1]
    return mostsim, cossim

In [187]:
X, vectorizer = vectorize_documents(documents=documents.data)
X_T = X.T

words = ["space", "computer", "religion", "game", "car"]
words_idx = [vectorizer.vocabulary_[p] for p in words]

for idx in words_idx:
    most_similar_idxs, cossim = most_similar_words(idx, X_T)

    print(f"La palabra '{idx2word[idx]}' tiene similitud con:")
    for similar_idx in most_similar_idxs:
        print(f"\tPalabra: {idx2word[similar_idx]} - Similitud: {cossim[similar_idx]:.3f}")
    print("\n")

La palabra 'space' tiene similitud con:
	Palabra: nasa - Similitud: 0.330
	Palabra: seds - Similitud: 0.297
	Palabra: shuttle - Similitud: 0.293
	Palabra: enfant - Similitud: 0.280
	Palabra: seti - Similitud: 0.246


La palabra 'computer' tiene similitud con:
	Palabra: decwriter - Similitud: 0.156
	Palabra: harkens - Similitud: 0.152
	Palabra: deluged - Similitud: 0.152
	Palabra: shopper - Similitud: 0.144
	Palabra: the - Similitud: 0.136


La palabra 'religion' tiene similitud con:
	Palabra: religious - Similitud: 0.245
	Palabra: religions - Similitud: 0.212
	Palabra: categorized - Similitud: 0.204
	Palabra: purpsoe - Similitud: 0.201
	Palabra: crusades - Similitud: 0.199


La palabra 'game' tiene similitud con:
	Palabra: games - Similitud: 0.212
	Palabra: espn - Similitud: 0.194
	Palabra: hockey - Similitud: 0.181
	Palabra: team - Similitud: 0.179
	Palabra: scored - Similitud: 0.171


La palabra 'car' tiene similitud con:
	Palabra: cars - Similitud: 0.180
	Palabra: criterium - Simili

📝 Observación: Para las palabras elegidas se muestran resultados con bastante relación semántica en la mayoria de los casos. Por ejemplo: game-espn (canal de deportes), car-civic (marca de automóviles), entre otros. Aunque existe un caso crítico que se observa con la palabra 'computer', la cual se relaciona con el stop-word 'the'. Esto demuestra una clara limitacion al usar este tipo de vectorización.